# Prodigal Gene Prediction

![Prodigal Gene Prediction](https://proto-bio.github.io/proto-assets/images/tool/prodigal/hero.png)

This notebook demonstrates `run_prodigal_prediction`, which identifies protein-coding genes in prokaryotic DNA sequences using Prodigal. It covers single-sequence prediction in meta mode, detailed ORF annotation inspection, batch processing, and result export in standard bioinformatics formats.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("prodigal")
display_overview("prodigal")
display_docs_section("prodigal", "Background")

# Prodigal

[Prodigal](https://github.com/hyattpd/Prodigal) is a [gene-prediction](https://en.wikipedia.org/wiki/Gene_prediction) program for [bacterial](https://en.wikipedia.org/wiki/Bacteria) and [archaeal](https://en.wikipedia.org/wiki/Archaea) genomes developed by Hyatt and colleagues at Oak Ridge National Laboratory. It predicts protein-coding genes using a [dynamic-programming](https://en.wikipedia.org/wiki/Dynamic_programming) algorithm that scores candidates by their coding potential, ribosome binding site strength, and [start codon](https://en.wikipedia.org/wiki/Start_codon) usage. This toolkit invokes Prodigal through the [pyrodigal](https://github.com/althonos/pyrodigal) Python interface and exposes a single registered tool that returns the predicted genes per input sequence together with their nucleotide and amino-acid sequences and Prodigal-specific annotations.

Prodigal ([Hyatt, Chen, LoCascio, Land, Larimer, and Hauser, 2010](https://doi.org/10.1186/1471-2105-11-119)) was developed as a fast and accurate replacement for earlier prokaryotic gene-prediction programs. The published method targets three specific objectives, namely improved gene-structure prediction, improved translation initiation site recognition, and reduction in the false-positive rate. The authors report that Prodigal achieves favourable results against the gene finders that were the established standard at the time of publication, and the program has since become one of the most widely used tools for automated prokaryotic genome annotation.

Prokaryotic gene prediction is straightforward relative to eukaryotic gene prediction because prokaryotic genes are contiguous, are not interrupted by [introns](https://en.wikipedia.org/wiki/Intron), and often begin with a [Shine-Dalgarno ribosome binding site](https://en.wikipedia.org/wiki/Shine-Dalgarno_sequence) located a short distance upstream of the start codon, typically around 5 to 10 nucleotides. Prodigal exploits these regularities by combining a dynamic-programming search across candidate [open reading frames](https://en.wikipedia.org/wiki/Open_reading_frame) with scoring terms for coding-region hexamer frequencies, the presence and strength of a recognised ribosome binding site motif, and the identity of the start codon. The program supports two operating modes. In single-genome mode it first trains its scoring parameters on the input sequence itself and then predicts genes using those trained parameters, which requires at least approximately 100 kilobases of input sequence for reliable training. In meta mode it applies a set of pre-trained parameters from a curated panel of reference genomes, which is appropriate for short contigs, draft assemblies, and [metagenomic](https://en.wikipedia.org/wiki/Metagenomics) samples.

This toolkit uses [pyrodigal](https://github.com/althonos/pyrodigal) ([Larralde, 2022](https://doi.org/10.21105/joss.04296)), a Python interface to Prodigal that exposes the original C implementation through Python bindings with SIMD-accelerated coding-region scoring. The interface reproduces the predictions of the reference Prodigal program while removing the need to manage an external command-line invocation.

## Available tools

In [2]:
display_available_tools("prodigal")

- **`run_prodigal_prediction()`** — Prokaryotic ORF and gene prediction using Prodigal

### `run_prodigal_prediction`

`run_prodigal_prediction` predicts protein-coding genes in prokaryotic DNA sequences using Prodigal via pyrodigal Python bindings. It runs in two modes: meta mode uses pre-trained metagenomic parameters and works on short contigs, incomplete assemblies, or mixed microbial samples; single-genome mode trains on the input sequence and requires at least 100 kb. Each predicted gene is returned as an ORF object with strand, reading frame, amino acid translation, nucleotide sequence, 1-indexed coordinates, GC content, start codon type, ribosome binding site motif and spacer, and partial-gene flags. Batch inputs are processed together and results are indexed by input sequence.

In [3]:
from proto_tools import ProdigalInput, ProdigalConfig, ProdigalOutput, run_prodigal_prediction

In [4]:
# Display input docs
display_api_reference("prodigal", "input", "run_prodigal_prediction")

# An E. coli lacZ gene fragment encoding beta-galactosidase
sequence = (
    "ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG"
    "GAAAACCCTGGCGTTACCCAACTTAATCGCCTTGCAGCACATCCCCCTTTC"
    "GCCAGCTGGCGTAATAGCGAAGAGGCCCGCACCGATCGCCCTTCCCAACAG"
    "TTGCGCAGCCTGAATGGCGAATGGCGCTTTGCCTGGTTTCCGGCACCAGAA"
    "GCGGTGCCGGAAAGCTGGCTGGAGTGCGATCTTCCTGAGGCCGATACTGTC"
    "GTCGTCCCCTCAAACTGGCAGATGCACGGTTACGATGCGCCCATCTACACC"
    "AACGTGACCTATCCCATTACGGTCAATCCGCCGTTTGTTCCCACGGAGAAT"
    "CCGACGGGTTGTTACTCGCTCACATTTAATGTTGATGAAAGCTGGCTACAG"
    "GAAGGCCAGACGCGAATTATTTTTGATGGCGTTAACTCGGCGTTTCATCTG"
    "TGGTGCAACGGGCGCTGGGTCGGTTACGGCCAGGACAGTCGTTTGCCGTCT"
    "TAA"
)
inputs = ProdigalInput(input_sequences=sequence)

**Input** — `ProdigalInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>input_sequences</code> | <code>list[str]</code> | required | DNA sequence(s) to analyze for open reading frames |

In [5]:
# Display config docs
display_api_reference("prodigal", "config", "run_prodigal_prediction")

# Meta mode with bacterial translation table | see docs above for all fields
config = ProdigalConfig(
    meta_mode=True,
    translation_table="bacterial",
    closed_ends=False,
)

**Config** — `ProdigalConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>meta_mode</code> | <code>bool</code> | <code>True</code> | Use meta mode for short sequences/fragments (True) or single-genome mode (False) |
| <code>translation_table</code> | <code>Literal['standard', 'vertebrate_mitochondrial', 'yeast_mitochondrial', 'mycoplasma', 'invertebrate_mitochondrial', 'ciliate_nuclear', 'echinoderm_mitochondrial', 'euplotid_nuclear', 'bacterial', 'alternative_yeast_nuclear', 'ascidian_mitochondrial', 'alternative_flatworm_mitochondrial', 'blepharisma_nuclear', 'chlorophycean_mitochondrial', 'trematode_mitochondrial', 'scenedesmus_mitochondrial', 'thraustochytrium_mitochondrial', 'rhabdopleuridae_mitochondrial', 'candidate_division_sr1']</code> | <code>'bacterial'</code> | NCBI genetic code for translation (bacterial = table 11, standard = table 1) |
| <code>closed_ends</code> | <code>bool</code> | <code>False</code> | Prevent genes from running off sequence edges (use True for complete circular genomes) |
| <code>mask</code> | <code>bool</code> | <code>False</code> | Treat runs of N as masked; do not call genes across them |
| <code>min_gene</code> | <code>int</code> | <code>90</code> | Minimum gene length in nt; lower for draft assemblies |
| <code>num_threads</code> | <code>int</code> | <code>192</code> | Number of threads for parallel processing (default: auto-detect all available cores) |

In [6]:
# Run the prediction
result = run_prodigal_prediction(inputs, config)

Running run_prodigal_prediction [00:00]

In [7]:
# Display output docs
display_api_reference("prodigal", "output", "run_prodigal_prediction")

# Inspect predicted genes
print(f"Found {result.num_orfs} predicted gene(s)")
for orf in result.predicted_orfs[0]:
    print(f"  {orf.id}: {orf.nucleotide_start}-{orf.nucleotide_end} ({orf.strand} strand, frame {orf.frame})")

# Inspect rich annotations on the first predicted ORF
orf = result.predicted_orfs[0][0]
print(f"\nGene annotations:")
print(f"  Start type:     {orf.start_type}")
print(f"  RBS motif:      {orf.rbs_motif}")
print(f"  RBS spacer:     {orf.rbs_spacer}")
print(f"  GC content:     {orf.gc_content:.3f}")
print(f"  Partial (5'/3'): {orf.partial_begin}/{orf.partial_end}")
print(f"  Protein length: {orf.amino_acid_length} aa")
print(f"  Protein:        {orf.amino_acid_sequence[:50]}...")

# Demonstrate batch processing
batch_sequences = [
    "ATGCGTAAATAA" * 50,
    "ATGGCATAA" * 50,
    "ATGAAACGT" * 50,
]
batch_result = run_prodigal_prediction(ProdigalInput(input_sequences=batch_sequences))
print(f"\nBatch: {batch_result.num_orfs} total genes across {len(batch_sequences)} sequences")
print(f"Genes per sequence: {batch_result.num_orfs_per_sequence}")

**Output** — `ProdigalOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>predicted_orfs</code> | <code>list[list[proto_tools.tools.orf_prediction.orf.ORF]]</code> | <code>[]</code> | List of ORF results per input sequence |

Found 1 predicted gene(s)
  seq_0_gene_1: 1-513 (+ strand, frame 1)

Gene annotations:
  Start type:     Edge
  RBS motif:      None
  RBS spacer:     None
  GC content:     55.361
  Partial (5'/3'): 1/0
  Protein length: 170 aa
  Protein:        MTMITDSLAVVLQRRDWENPGVTQLNRLAAHPPFASWRNSEEARTDRPSQ...


Running run_prodigal_prediction [00:00]


Batch: 3 total genes across 3 sequences
Genes per sequence: [1, 1, 1]


## Export Results

Predicted genes can be exported to GFF (standard gene annotation format), CSV or JSON for tabular analysis, or FASTA format (FAA for protein sequences, FNA for nucleotide sequences) for downstream tools such as BLAST or multiple sequence alignment.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("prodigal_results", export_path=output_dir, file_format="gff")
print(f"Exported GFF to {output_dir / 'prodigal_results.gff'}")

result.export("prodigal_results", export_path=output_dir, file_format="csv")
print(f"Exported CSV to {output_dir / 'prodigal_results.csv'}")

result.export("prodigal_results", export_path=output_dir, file_format="faa")
print(f"Exported protein FASTA to {output_dir / 'prodigal_results.faa'}")

Exported GFF to example_output/prodigal_results.gff


Exported CSV to example_output/prodigal_results.csv
Exported protein FASTA to example_output/prodigal_results.faa
